# Neden Sayıya Çevir?

- ML modelleri sayı ister, metin almaz

    - Çözüm metin -> vektör


# BoW Fikri

- Sırayı unut, kelime frekansını say

- Belgeleri torbaya koyduğun kelimelerin listesi gib düşün.

- "bugun çok güzel" ve "güzel çok bugun" BoW açısından AYNI

1. Bag Of Words CountVectorizer, n-gram, sınıflandırma örneği

In [1]:
# 3 BELGE ÖRNEĞİ
docs = [
    "NLP çok eğlenceli bir alan",
    "Makine öğrenmesi ve NLP birlikte güçlü",
    "Python NLP için mükemmel"
]

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer - temel kullanım
vec = CountVectorizer()
X = vec.fit_transform(docs)

print(f" Vacabulary ({len(vec.get_feature_names_out())} kelime)")
print(vec.get_feature_names_out())

print(f"\nMatrix shape: {X.shape}")
print(f"Matrix (dense):\n{X.toarray()}")

# Satır = belge, sütun = kelime, hücre = frekans

 Vacabulary (13 kelime)
['alan' 'bir' 'birlikte' 'eğlenceli' 'güçlü' 'için' 'makine' 'mükemmel'
 'nlp' 'python' 've' 'çok' 'öğrenmesi']

Matrix shape: (3, 13)
Matrix (dense):
[[1 1 0 1 0 0 0 0 1 0 0 1 0]
 [0 0 1 0 1 0 1 0 1 0 1 0 1]
 [0 0 0 0 0 1 0 1 1 1 0 0 0]]


In [4]:
import pandas as pd

df = pd.DataFrame(X.toarray(), columns= vec.get_feature_names_out(),
                  index = [f"Belge {i + 1}" for i in range(len(docs))])
print(df)

         alan  bir  birlikte  eğlenceli  güçlü  için  makine  mükemmel  nlp  \
Belge 1     1    1         0          1      0     0       0         0    1   
Belge 2     0    0         1          0      1     0       1         0    1   
Belge 3     0    0         0          0      0     1       0         1    1   

         python  ve  çok  öğrenmesi  
Belge 1       0   0    1          0  
Belge 2       0   1    0          1  
Belge 3       1   0    0          0  


## Sparse Matrix
- Matrix 10.000 x 100.000 = 1 milyar hücre

- Çoğu 0 (her belge sadece birkaç yüz kelime içerir) 

- Sparse matrix kullan (spicy.sparse)

- Sadece sıfır olmayanları sakla -> bellek verimli

## BoW Sınırları

- Sıra kaybolur: "iyi değil"  = "değil iyi"

- Semantik yok: "mutlu" ve "sevinçli" farklı feature

- Yaygın kelimeler baskın: "ve", "bir" her belgede

- Nadir Kelimeler kaybolur: vocab büyüklüğüne takılır


# Hiperparametreler

- max_features -> En sık N kelime

- min_df -> En az N belgede geçsin

- max_df -> En fazla N belgede geçsin (stopword filtre)

- ngram_range -> (1,2) = unigram + bigram

- stop_words -> "english" veya özel liste

- lowercase -> Default True

- token_pattern -> Regex(default \b\w\w+\b)

In [ ]:
# Hiperparametreler

# max_features: en sık kullanılan N kelime
vec_limited = CountVectorizer(max_features = 5)
X_limited = vec_limited.fit_transform(docs)
print(f"\nTop 5 kelime: {vec_limited.get_feature_names_out()}")

# min_df: Kelime en az N belgede geçsin
vec_mindf = CountVectorizer(min_df=2) # en az 2 belgede
X_mindf = vec_mindf.fit_transform(docs)
print(f"\nmin_df = 2: {vec_mindf.get_feature_names_out()}")

vec_sw = CountVectorizer(stop_words= ["ve","için","bir"])
X_sw = vec_sw.fit_transform(docs)
print(f"\nNo SW: {vec_sw.get_feature_names_out()}")


Top 5 kelime: ['alan' 'bir' 'birlikte' 'eğlenceli' 'nlp']

min_df = 2: ['nlp']

No SW: ['alan' 'birlikte' 'eğlenceli' 'güçlü' 'makine' 'mükemmel' 'nlp' 'python'
 'çok' 'öğrenmesi']


# N-gram - Sıralama Bilgisi
- vec = CountVectorizer(ngram_range = (1,2)) # unigram + bigram
- X = vec.fit_transform(["new york city güzel])

vec.get_feature_names_out()
- ["city","güzel","new","new york","york","york city"]

In [7]:
# N-GRAM - tek kelime yerine kelime çiftleri
# Bigram: 2 kelimelik ardışık gruplar
vec_bi = CountVectorizer(ngram_range= (1,2)) # unigram + bigram
X_bi = vec_bi.fit_transform(docs)

print(f"\nUnigram + Bigram features")
for f in vec_bi.get_feature_names_out():
    print(f"    {f}")

# "makina öğrenmesi" şimdi tek feeature - anlam korunuyor


Unigram + Bigram features
    alan
    bir
    bir alan
    birlikte
    birlikte güçlü
    eğlenceli
    eğlenceli bir
    güçlü
    için
    için mükemmel
    makine
    makine öğrenmesi
    mükemmel
    nlp
    nlp birlikte
    nlp için
    nlp çok
    python
    python nlp
    ve
    ve nlp
    çok
    çok eğlenceli
    öğrenmesi
    öğrenmesi ve


In [12]:
from sklearn.naive_bayes import MultinomialNB

# Gerçek SINIFLANDIRMA - SPAM Filtresi

mails = [
    "ücretsiz hediye kazandınız tıkla", # spam
    "acele 100 tl çekiliş kaçırma", # spam
    "kazandınız hemen para çek",    # spam
    "viagra kilo ver hap hediye",   # spam
    "toplantı yarın saat 14 te önemli", # ham
    "rapor ekte inceleyin", # ham
    "müşteri sözleşme imza",    # ham
    "proje teslim tarih yaklaştı"   # ham
]
labels = [1,1,1,1,0,0,0,0]

# BoW
vec = CountVectorizer()
X = vec.fit_transform(mails)

# Model
model = MultinomialNB().fit(X,labels)

# Test
test_mails = [
    "kazandınız ücretsiz çekiliş", # spam beklenen
    "rapor incele toplantı"     # ham beklenen
]
X_test = vec.transform(test_mails)
preds = model.predict(X_test)
probas = model.predict_proba(X_test)

# 1. test maili
mail,pred,proba = test_mails[0], preds[0], probas[0]
label = "SPAM" if pred == 1 else "HAM"
print(f"{label} ({proba[pred]:.2%}): {mail}")

# 2. test maili
mail,pred,proba = test_mails[1], preds[1], probas[1]
label = "SPAM" if pred == 1 else "HAM"
print(f"{label} ({proba[pred]:.2%}): {mail}")

SPAM (91.39%): kazandınız ücretsiz çekiliş
HAM (81.27%): rapor incele toplantı
